In [1]:
!pip install -q chromadb pypdf requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [2]:
import chromadb
from chromadb.utils import embedding_functions
import os
import requests
from pypdf import PdfReader

In [3]:
client = chromadb.Client()

print("ChromaDB client created successfully")

ChromaDB client created successfully


In [4]:
embedding_function = embedding_functions.DefaultEmbeddingFunction()

print("Embedding function created successfully")

Embedding function created successfully


In [5]:
collection = client.get_or_create_collection(
    name="w5d4_documents",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"}
)

print("Collection created:", collection.name)

Collection created: w5d4_documents


In [6]:
documents = [
    "Machine learning allows computers to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Artificial intelligence is used in healthcare for diagnosis.",
    "Natural language processing helps computers understand human language.",
    "Computer vision allows machines to understand images and videos.",
    "Python is widely used for artificial intelligence and machine learning.",
    "Data preprocessing improves the quality of machine learning models.",
    "Feature engineering creates useful input features from raw data.",
    "Classification predicts categories or classes from input data.",
    "Regression predicts continuous numerical values.",
    "ChromaDB is a vector database used for storing embeddings.",
    "Semantic search retrieves documents based on meaning.",
    "Embeddings represent text as numerical vectors.",
    "Cosine similarity measures similarity between vectors.",
    "Large language models can generate human-like text.",
    "Ollama allows local execution of large language models.",
    "Retrieval augmented generation combines retrieval with generation.",
    "PDF documents can be converted into text for semantic search.",
    "Vector databases are useful for building question-answering systems.",
    "AI applications can combine vector databases with language models."
]

ids = [f"doc_{i}" for i in range(20)]

In [7]:
metadatas = [
    {"category": "AI"} for _ in range(20)
]

for i in range(10, 15):
    metadatas[i]["category"] = "Database"

for i in range(15, 20):
    metadatas[i]["category"] = "GenAI"

print("Metadata created")

Metadata created


In [8]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print("Number of documents:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s]


Number of documents: 20


In [9]:
query = "How do computers understand and process human language?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print("Query:", query)
print("\nTop 3 results:\n")

for i, doc in enumerate(results["documents"][0]):
    print(f"{i+1}. {doc}")

Query: How do computers understand and process human language?

Top 3 results:

1. Natural language processing helps computers understand human language.
2. Computer vision allows machines to understand images and videos.
3. Large language models can generate human-like text.


In [10]:
print("Distances:")

for i, distance in enumerate(results["distances"][0]):
    print(f"Result {i+1}: {distance}")

Distances:
Result 1: 0.28668665885925293
Result 2: 0.5250408053398132
Result 3: 0.5588231086730957


In [11]:
filtered_results = collection.query(
    query_texts=["vector databases and embeddings"],
    n_results=3,
    where={"category": "Database"}
)

print("Filtered results:\n")

for doc in filtered_results["documents"][0]:
    print("-", doc)

Filtered results:

- ChromaDB is a vector database used for storing embeddings.
- Embeddings represent text as numerical vectors.
- Cosine similarity measures similarity between vectors.


In [12]:
print("Manual verification:\n")

for i, doc in enumerate(filtered_results["documents"][0]):
    print(f"{i+1}. {doc}")

Manual verification:

1. ChromaDB is a vector database used for storing embeddings.
2. Embeddings represent text as numerical vectors.
3. Cosine similarity measures similarity between vectors.


In [13]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

reader = PdfReader(pdf_path)

pdf_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        pdf_text += text + "\n"

print("PDF loaded successfully")
print("Characters extracted:", len(pdf_text))

Saving W5D3_AI_ML_Retrieval_Document.pdf to W5D3_AI_ML_Retrieval_Document.pdf
PDF loaded successfully
Characters extracted: 3589


In [14]:
chunk_size = 800

chunks = [
    pdf_text[i:i + chunk_size]
    for i in range(0, len(pdf_text), chunk_size)
]

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk[:300])

Number of chunks: 5

--- Chunk 1 ---
AI and Machine Learning: A Short Reference Document
1. Machine Learning
Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and
make predictions or decisions without being explicitly programmed for every situation. A machine learning
workflow us

--- Chunk 2 ---
g works with unlabeled data. The algorithm attempts to discover useful patterns or
structures within the data. Clustering is a common example. It groups similar observations together based on
their features.
4. Deep Learning
Deep learning is a part of machine learning that uses neural networks conta

--- Chunk 3 ---
the most informative features.
6. Model Evaluation
A model should be evaluated on data that was not used to train it. Common evaluation methods include
train-test splitting and cross-validation. Classification metrics include accuracy, precision, recall, and
F1-score. For regression, common metrics 


In [15]:
pdf_collection = client.get_or_create_collection(
    name="pdf_collection",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"}
)

pdf_ids = [f"pdf_chunk_{i}" for i in range(len(chunks))]

pdf_metadata = [
    {"source": pdf_path, "chunk": i}
    for i in range(len(chunks))
]

pdf_collection.add(
    ids=pdf_ids,
    documents=chunks,
    metadatas=pdf_metadata
)

print("PDF chunks stored:", pdf_collection.count())

PDF chunks stored: 5


In [16]:
question = "What is the main topic discussed in this document?"

pdf_results = pdf_collection.query(
    query_texts=[question],
    n_results=3
)

top_chunks = pdf_results["documents"][0]

print("Top 3 retrieved chunks:\n")

for i, chunk in enumerate(top_chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk[:700])

Top 3 retrieved chunks:


--- Chunk 1 ---
tions about text. In a RAG system, an LLM can use retrieved
document chunks as context before generating an answer.
Purpose of this document
This document is provided as a sample PDF for the W5D3 ChromaDB practical. It can be used to
demonstrate PDF text extraction, document chunking, vector storage, similarity retrieval, and passing
retrieved context to a local Ollama language model.



--- Chunk 2 ---
language model. Documents
are divided into smaller chunks and stored in a vector database. When a user asks a question, the system
retrieves the most relevant chunks and provides them as context to a language model. This can help the
model answer questions using information from a specific document.
9. Vector Databases and ChromaDB
A vector database stores representations of information that can be compared for similarity. ChromaDB is a
vector database commonly used for storing documents and retrieving relevant information. Similarity search
can

In [17]:
context = "\n\n".join(top_chunks)

prompt = f"""
Answer the question using only the context provided below.

Context:
{context}

Question:
{question}

Answer:
"""

print(prompt)


Answer the question using only the context provided below.

Context:
tions about text. In a RAG system, an LLM can use retrieved
document chunks as context before generating an answer.
Purpose of this document
This document is provided as a sample PDF for the W5D3 ChromaDB practical. It can be used to
demonstrate PDF text extraction, document chunking, vector storage, similarity retrieval, and passing
retrieved context to a local Ollama language model.



language model. Documents
are divided into smaller chunks and stored in a vector database. When a user asks a question, the system
retrieves the most relevant chunks and provides them as context to a language model. This can help the
model answer questions using information from a specific document.
9. Vector Databases and ChromaDB
A vector database stores representations of information that can be compared for similarity. ChromaDB is a
vector database commonly used for storing documents and retrieving relevant information. Similarit

In [19]:
!sudo apt-get update -qq
!sudo apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 28 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (726 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controllin

In [20]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [21]:
!ollama --version

In [22]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [23]:
!curl http://localhost:11434/api/tags

{"models":[]}

In [24]:
!ollama pull llama3.2:3b

In [25]:
!ollama list

NAME           ID              SIZE      MODIFIED       
llama3.2:3b    a80c4f17acd5    2.0 GB    33 seconds ago    


In [26]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "What is ChromaDB?",
        "stream": False
    }
)

print(response.status_code)
print(response.json()["response"])

200
I couldn't find any information on "ChromaDB". It's possible that it's a relatively unknown or niche project, or it could be a misspelling or variation of a different term.

However, I did find information on a database called "Chroma", which is an open-source, cloud-based database designed for storing, retrieving, and managing metadata for audio and other media files. Chroma is often used in audio editing, metadata management, and other applications that require efficient storage and retrieval of metadata.

If you have any more information or context about ChromaDB, I may be able to provide more specific information or guidance.


In [27]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": prompt,
        "stream": False
    }
)

print(response.json()["response"])

The main topic discussed in this document is a system for asking questions to a large language model (LLM) using a vector database, specifically ChromaDB, and retrieving relevant document chunks as context before generating an answer.


In [29]:
print("Question:")
print(question)

print("\nRetrieved chunks:")
for i, chunk in enumerate(top_chunks):
    print(f"{i+1}. {chunk[:200]}...")

print("\nFinal Ollama Answer:")
print(response.json()["response"])

Question:
What is the main topic discussed in this document?

Retrieved chunks:
1. tions about text. In a RAG system, an LLM can use retrieved
document chunks as context before generating an answer.
Purpose of this document
This document is provided as a sample PDF for the W5D3 Chro...
2. language model. Documents
are divided into smaller chunks and stored in a vector database. When a user asks a question, the system
retrieves the most relevant chunks and provides them as context to a ...
3. g works with unlabeled data. The algorithm attempts to discover useful patterns or
structures within the data. Clustering is a common example. It groups similar observations together based on
their fe...

Final Ollama Answer:
The main topic discussed in this document is a system for asking questions to a large language model (LLM) using a vector database, specifically ChromaDB, and retrieving relevant document chunks as context before generating an answer.
